# Extraire les entités de la presse de 1914

Quelles personnes, quels lieux, quelles organisations un passage nomme-t-il ? Trois méthodes, du plus rapide au plus souple, comparées sur 40 passages annotés par un LLM, chaque entité vérifiée dans son texte :

1. spaCy, ses étiquettes fixes
2. GLiNER, des étiquettes au choix
3. Un LLM qui remplit un schéma
4. Relier les noms à une liste de référence
5. Comparer, et repérer les inventions

Ce notebook va plus vite avec un GPU : *Exécution > Modifier le type d'exécution > GPU T4*. Sans GPU, il tourne quand même : un LLM plus petit (Qwen3-1.7B au lieu de Qwen3-4B), plus lent sur un processeur, n'extrait alors que 5 des 40 passages ; la suite charge les extractions des 40 faites par Qwen3-4B sur GPU.

La limite : cette référence vient elle-même d'un LLM. Elle dit quelles entités sont bien dans le texte, pas lesquelles manquent, et elle peut avantager un LLM sur spaCy et GLiNER. Pour un projet réel, annotez à la main.

## Données et outils

Le LLM tourne dans [Ollama](https://ollama.com), sur la machine Colab : la cellule suivante l'installe et télécharge le modèle en arrière-plan, pendant que vous travaillez sur spaCy et GLiNER.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-extraction/gold.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-extraction/teacher-gold.jsonl
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
%pip install -q gliner rapidfuzz
!python -m spacy download fr_core_news_md > /dev/null

import os
import subprocess

import torch

GPU = torch.cuda.is_available()
DEVICE = "cuda" if GPU else "cpu"
# Sur GPU, Qwen3-4B. Sur CPU, Qwen3-1.7B, deux fois plus rapide : un modèle « hybride »,
# qui réfléchit avant de répondre, sauf avec reasoning_effort="none" (dans les appels).
MODELE = "qwen3:4b-instruct-2507-q4_K_M" if GPU else "qwen3:1.7b"
# Sur GPU, 8 demandes à la fois ; sur CPU, une : le début commun des prompts (les
# consignes) n'est alors calculé qu'une fois.
paralleles = "8" if GPU else "1"
env = {**os.environ, "OLLAMA_NUM_PARALLEL": paralleles, "OLLAMA_KEEP_ALIVE": "-1"}
subprocess.Popen(
  ["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
telechargement = subprocess.Popen(
  f"sleep 5 && ollama pull {MODELE}",
  shell=True,
  stdout=subprocess.DEVNULL,
  stderr=subprocess.DEVNULL,
)

In [ ]:
import json

import pandas as pd
from rapidfuzz import fuzz

gold = pd.read_json("gold.jsonl", lines=True)
CHAMPS = ["people", "places", "organizations"]
print(len(gold), "passages annotés")
gold.head(3)

`evalue(predictions)` compare des extractions (une liste de dictionnaires `{"people": [...], "places": [...], "organizations": [...]}`, dans l'ordre de `gold`) aux annotations : pour chaque champ, la précision (la part des entités extraites qui sont justes), le rappel (la part des entités attendues qui sont trouvées) et leur F1. La correspondance est souple : deux noms correspondent si leur similarité (`fuzz.ratio`, une distance d'édition sans substitution) dépasse 85 sur 100, ou si l'un contient l'autre (« Jaurès » et « Jean Jaurès »).

In [ ]:
def normalise(nom: str) -> str:
  return " ".join(nom.lower().replace("’", "'").split())


def correspond(a: str, b: str) -> bool:
  a, b = normalise(a), normalise(b)
  return fuzz.ratio(a, b) >= 85 or (min(len(a), len(b)) >= 4 and (a in b or b in a))


def evalue(predictions: list[dict[str, list[str]]]) -> pd.DataFrame:
  scores = {}
  for champ in CHAMPS:
    trouves = attendus = bons_trouves = bons_attendus = 0
    for prediction, (_, ligne) in zip(predictions, gold.iterrows()):
      predits = list(dict.fromkeys(prediction.get(champ, [])))
      verite = ligne[champ]
      trouves += len(predits)
      attendus += len(verite)
      bons_trouves += sum(any(correspond(p, t) for t in verite) for p in predits)
      bons_attendus += sum(any(correspond(p, t) for p in predits) for t in verite)
    precision = bons_trouves / max(trouves, 1)
    rappel = bons_attendus / max(attendus, 1)
    scores[champ] = {
      "précision": precision,
      "rappel": rappel,
      "F1": 2 * precision * rappel / max(precision + rappel, 1e-9),
    }
  return pd.DataFrame(scores).T.round(2)


resultats = {}

## 1. spaCy

Chargez `fr_core_news_md` et extrayez les entités de chaque passage : `PER` pour les personnes, `LOC` pour les lieux, `ORG` pour les organisations. Évaluez, et rangez le tableau dans `resultats["spaCy"]`. Combien de temps pour les 40 passages ?

**Documentation utile**

- [`Language.pipe`](https://spacy.io/api/language#pipe)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `spacy.load("fr_core_news_md")`, puis `nlp.pipe(gold.text)` : un `Doc` par passage, plus vite qu'un par un
- `doc.ents` : les entités d'un passage ; `ent.text` est leur texte, `ent.label_` leur étiquette (`PER`, `LOC`, `ORG`)
- le format attendu par `evalue` : une liste de dictionnaires `{"people": [...], "places": [...], "organizations": [...]}`, dans l'ordre de `gold`
- `time.time()` avant et après pour chronométrer

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
import time

import spacy

nlp = spacy.load(...)
ETIQUETTES = {"PER": "people", "LOC": "places", "ORG": "organizations"}
debut = time.time()
predictions_spacy = []
for doc in nlp.pipe(...):
  prediction = {champ: [] for champ in CHAMPS}
  for ent in doc.ents:
    if ent.label_ in ETIQUETTES:
      prediction[ETIQUETTES[ent.label_]].append(...)
  predictions_spacy.append(prediction)
print(f"{time.time() - debut:.1f} s")
resultats["spaCy"] = evalue(...)
resultats["spaCy"]
```

### Solution

In [ ]:
import time

import spacy

nlp = spacy.load("fr_core_news_md")
ETIQUETTES = {"PER": "people", "LOC": "places", "ORG": "organizations"}
debut = time.time()
predictions_spacy = []
for doc in nlp.pipe(gold.text):
  prediction = {champ: [] for champ in CHAMPS}
  for ent in doc.ents:
    if ent.label_ in ETIQUETTES:
      prediction[ETIQUETTES[ent.label_]].append(ent.text)
  predictions_spacy.append(prediction)
print(f"{time.time() - debut:.1f} s")
resultats["spaCy"] = evalue(predictions_spacy)
resultats["spaCy"]

## 2. GLiNER

[GLiNER](https://github.com/urchade/GLiNER) trouve les entités des étiquettes qu'on lui donne, sans entraînement. Chargez `urchade/gliner_multi-v2.1` (sur le GPU s'il y en a un : `DEVICE`) et extrayez les entités des étiquettes `["personne", "lieu", "organisation"]` (`predict_entities`). Évaluez, et rangez le tableau dans `resultats["GLiNER"]`, les extractions dans `predictions_gliner` (l'exercice 4 s'en sert). Puis essayez d'autres étiquettes (« journal », « parti politique »…, rangées dans `organizations`) ou un autre seuil : quels champs y gagnent ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `GLiNER.from_pretrained("urchade/gliner_multi-v2.1").to(DEVICE)` : charge le modèle, puis le place sur le GPU (`DEVICE` vaut `"cpu"` sans GPU)
- `gliner.predict_entities(texte, étiquettes, threshold=0.4)` : renvoie une liste de dictionnaires, avec le texte de l'entité (`"text"`) et son étiquette (`"label"`) ; `threshold` est la confiance minimale (0,5 par défaut) ; 0,4, une valeur de référence pour comparer les essais entre eux, n'est pas forcément le meilleur seuil : à vous de le chercher en le faisant varier
- un dictionnaire `CHAMP_DE = {"personne": "people", ...}` convertit chaque étiquette de GLiNER en champ de `evalue`
- une fonction `extrait_gliner(champ_de, seuil)` qui renvoie les extractions des 40 passages : un second essai n'est plus qu'un appel
- `{**CHAMP_DE, "journal": "organizations"}` : le même dictionnaire, avec une étiquette de plus

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from gliner import GLiNER

gliner = GLiNER.from_pretrained(...).to(DEVICE)


def extrait_gliner(
  champ_de: dict[str, str], seuil: float
) -> list[dict[str, list[str]]]:
  predictions = []
  for texte in gold.text:
    prediction = {champ: [] for champ in CHAMPS}
    for entite in gliner.predict_entities(texte, list(champ_de), threshold=seuil):
      prediction[champ_de[entite["label"]]].append(entite[...])
    predictions.append(prediction)
  return predictions


CHAMP_DE = {"personne": "people", "lieu": "places", "organisation": "organizations"}
debut = time.time()
predictions_gliner = extrait_gliner(CHAMP_DE, seuil=...)
print(f"{time.time() - debut:.1f} s")
resultats["GLiNER"] = evalue(...)
print(resultats["GLiNER"])

# Un second essai : d'autres étiquettes, ou un autre seuil
autres_etiquettes = {**CHAMP_DE, ...: "organizations"}
evalue(extrait_gliner(autres_etiquettes, seuil=...))
```

### Solution

In [ ]:
from gliner import GLiNER

gliner = GLiNER.from_pretrained("urchade/gliner_multi-v2.1").to(DEVICE)


def extrait_gliner(
  champ_de: dict[str, str], seuil: float
) -> list[dict[str, list[str]]]:
  predictions = []
  for texte in gold.text:
    prediction = {champ: [] for champ in CHAMPS}
    for entite in gliner.predict_entities(texte, list(champ_de), threshold=seuil):
      prediction[champ_de[entite["label"]]].append(entite["text"])
    predictions.append(prediction)
  return predictions


CHAMP_DE = {"personne": "people", "lieu": "places", "organisation": "organizations"}
debut = time.time()
predictions_gliner = extrait_gliner(CHAMP_DE, seuil=0.4)
print(f"{time.time() - debut:.1f} s")
resultats["GLiNER"] = evalue(predictions_gliner)
print(resultats["GLiNER"])

# Changer d'étiquettes ne demande aucun entraînement : c'est ce que spaCy, aux étiquettes
# fixées à l'entraînement, ne peut pas faire. Avec « journal » et « parti politique »,
# organizations passe de 0,60 à 0,70 de F1, et people de 0,73 à 0,78. Un seuil de 0,6
# fait aussi mieux que 0,4 avec les trois étiquettes (people 0,77, places 0,84), mais
# pas avec les cinq (places 0,78) : les réglages se mesurent ensemble, sur vos données.
# Le tableau de l'exercice 5 garde le premier essai.
autres_etiquettes = {
  **CHAMP_DE,
  "journal": "organizations",
  "parti politique": "organizations",
}
evalue(extrait_gliner(autres_etiquettes, seuil=0.4))

## 3. Un LLM qui remplit un schéma

Le modèle tourne dans Ollama, qui offre une API compatible OpenAI : le même code fonctionnerait avec une API commerciale, en changeant l'adresse, la clé et le nom du modèle.

Décrivez le résultat attendu avec une classe Pydantic `Entites` : trois listes de chaînes, et une date qui peut valoir `None` (`str | None`), que le modèle laisse vide plutôt que de l'inventer. Puis écrivez `extrait(texte)` qui demande au modèle les entités du passage, avec `client.chat.completions.parse` : la réponse respecte le schéma, sauf si elle est coupée (`max_tokens`) ou refusée. Votre prompt doit donner les consignes d'annotation (ci-dessous) et le format de la date (ISO, `null` si le passage n'en donne pas). Lancez l'extraction des passages de `textes_llm` (les 40 ; sans GPU, les 5 premiers) **en parallèle**, mesurez le temps, évaluez. La date ne s'évalue pas (`gold.jsonl` n'en a pas) : lisez-la sur quelques passages, face à la date du numéro, dans leur identifiant (`gold.id`).

Consignes d'annotation : chaque entité nommée une fois, telle qu'écrite dans le texte (erreurs d'OCR comprises) ; les personnes sans leur titre (M., Mme, sir) ; les lieux (pays, villes, rues, bâtiments) ; les organisations nommées (assemblées, partis, syndicats, journaux, agences) ; pas de mentions génériques (« le gouvernement »).

**Documentation utile**

- [`BaseModel.model_dump`](https://docs.pydantic.dev/latest/api/base_model/#pydantic.BaseModel.model_dump)
- [`ThreadPoolExecutor.map`](https://docs.python.org/3/library/concurrent.futures.html#concurrent.futures.Executor.map)

In [ ]:
telechargement.wait()  # le modèle est téléchargé

from openai import OpenAI
from pydantic import BaseModel

# timeout : sans GPU, Ollama traite les requêtes une à une, et la dernière des requêtes
# envoyées ensemble attend toutes les autres (le délai par défaut est de 600 s).
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama", timeout=1800)
# Un premier appel charge le modèle (sur le GPU s'il y en a un), où il reste : avant les calculs en parallèle.
client.chat.completions.create(
  model=MODELE, messages=[{"role": "user", "content": "Bonjour"}], max_tokens=1
)
print(MODELE, "prêt")

# Les passages que le LLM traite : les 40 sur GPU ; sans GPU, 5 (une dizaine de minutes).
textes_llm = gold.text if GPU else gold.text[:5]

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `class Entites(BaseModel)` : trois champs, `people`, `places` et `organizations`, de type `list[str]`, et `date`, de type `str | None` (`None` si le passage n'en donne pas)
- `client.chat.completions.parse(model=MODELE, messages=[{"role": "system", "content": CONSIGNES}, {"role": "user", "content": texte}], response_format=Entites, temperature=0, reasoning_effort="none")`
- `reasoning_effort="none"` : le modèle répond sans phase de réflexion (sans GPU, Qwen3-1.7B en aurait une, qui multiplie le temps de réponse)
- `reponse.choices[0].message.parsed.model_dump()` : l'objet lu, converti en dictionnaire
- `ThreadPoolExecutor(8)` : 8 appels en parallèle ; `list(executeur.map(extrait, textes_llm))` renvoie les extractions dans l'ordre des passages
- `zip(predictions_llm[:5], gold.id)` : les 5 premières extractions, avec l'identifiant de leur passage, qui contient la date du numéro

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from concurrent.futures import ThreadPoolExecutor


class Entites(BaseModel):
  people: list[str]
  places: ...
  organizations: ...
  date: ...


CONSIGNES = """Extrais les entités nommées de ce passage de presse de 1914.
- people : les personnes, sans leur titre (M., Mme, sir, lord) ;
- places : les lieux (pays, régions, villes, rues, bâtiments) ;
- organizations : les organisations nommées (assemblées, partis, syndicats, journaux, agences).
- date : la date des faits, au format ISO (AAAA-MM-JJ), null si le passage n'en donne pas.
Chaque entité une fois, telle qu'elle est écrite dans le texte, erreurs comprises.
Pas de mentions génériques comme « le gouvernement ». N'invente rien."""


def extrait(texte: str) -> dict[str, list[str] | str | None]:
  reponse = client.chat.completions.parse(
    model=MODELE,
    messages=[{"role": "system", "content": ...}, {"role": "user", "content": ...}],
    response_format=...,
    temperature=0,
    reasoning_effort="none",
  )
  return reponse.choices[0].message.parsed.model_dump()


debut = time.time()
with ThreadPoolExecutor(8) as executeur:
  predictions_llm = list(executeur.map(..., textes_llm))
print(f"{time.time() - debut:.1f} s")
# La date, face à celle du numéro (dans l'identifiant du passage)
for prediction, id_passage in zip(predictions_llm[:5], gold.id):
  print(f"{id_passage:26} {prediction[...]}")
resultats["LLM"] = evalue(...)
resultats["LLM"]
```

### Solution

In [ ]:
from concurrent.futures import ThreadPoolExecutor


class Entites(BaseModel):
  people: list[str]
  places: list[str]
  organizations: list[str]
  date: str | None


CONSIGNES = """Extrais les entités nommées de ce passage de presse de 1914.
- people : les personnes, sans leur titre (M., Mme, sir, lord) ;
- places : les lieux (pays, régions, villes, rues, bâtiments) ;
- organizations : les organisations nommées (assemblées, partis, syndicats, journaux, agences).
- date : la date des faits, au format ISO (AAAA-MM-JJ), null si le passage n'en donne pas.
Chaque entité une fois, telle qu'elle est écrite dans le texte, erreurs comprises.
Pas de mentions génériques comme « le gouvernement ». N'invente rien."""


def extrait(texte: str) -> dict[str, list[str] | str | None]:
  reponse = client.chat.completions.parse(
    model=MODELE,
    messages=[
      {"role": "system", "content": CONSIGNES},
      {"role": "user", "content": texte},
    ],
    response_format=Entites,
    temperature=0,
    reasoning_effort="none",
  )
  return reponse.choices[0].message.parsed.model_dump()


debut = time.time()
with ThreadPoolExecutor(8) as executeur:
  predictions_llm = list(executeur.map(extrait, textes_llm))
print(f"{time.time() - debut:.1f} s")
# La date ne s'évalue pas (gold.jsonl n'en a pas) : on la lit, face à la date du
# numéro, dans l'identifiant du passage.
for prediction, id_passage in zip(predictions_llm[:5], gold.id):
  print(f"{id_passage:26} {prediction['date']}")
resultats["LLM"] = evalue(predictions_llm)
resultats["LLM"]

## 4. Relier les noms à une liste de référence

Les mêmes personnes apparaissent sous des formes différentes (« Jaurès », « Jean Jaurès », « M. Jaurès »), parfois mal reconnues par l'OCR. Reliez les personnes trouvées par GLiNER (`predictions_gliner`, les 40 passages, avec ou sans GPU) à `REFERENCE` : pour chacune, le nom le plus proche (`rapidfuzz.process.extractOne`), gardé si son score atteint un seuil de 85, puis de 75. Que gagne-t-on, que perd-on ? Trouvez un faux lien, et une règle qui l'écarte.

**Documentation utile**

- [`rapidfuzz.process.extractOne`](https://rapidfuzz.github.io/RapidFuzz/Usage/process.html#extractone)
- [`rapidfuzz.fuzz.WRatio`](https://rapidfuzz.github.io/RapidFuzz/Usage/fuzz.html#wratio)
- [`rapidfuzz.fuzz.ratio`](https://rapidfuzz.github.io/RapidFuzz/Usage/fuzz.html#ratio)

In [ ]:
REFERENCE = [
  "Jean Jaurès",
  "Joseph Caillaux",
  "Henriette Caillaux",
  "Gaston Calmette",
  "Raymond Poincaré",
  "René Viviani",
  "Georges Clemenceau",
  "Jules Guesde",
  "Léon Jouhaux",
  "Édouard Vaillant",
  "Edward Grey",
  "Herbert Asquith",
  "Paul Cambon",
  "Raoul Villain",
  "Gabriel Hanotaux",
  "Antonino di San Giuliano",
  "Émile Vandervelde",
  "John Simon",
  "Woodrow Wilson",
]

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `process.extractOne(nom, REFERENCE, scorer=fuzz.WRatio)` renvoie un tuple `(choix, score, position)` ; `WRatio` combine plusieurs mesures et tolère l'ordre des mots et les noms partiels
- `sorted({nom for prediction in predictions_gliner for nom in prediction["people"]})` : les personnes extraites par GLiNER, sans doublon
- pour la règle : la longueur du nom extrait, ou son dernier mot (`nom.split()[-1]`) comparé à celui du nom de référence avec `fuzz.ratio`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from rapidfuzz import process

personnes = sorted({nom for prediction in ... for nom in prediction["people"]})
for seuil in (85, 75):
  print(f"--- seuil de {seuil}")
  for nom in personnes:
    correspondance, score, _ = process.extractOne(nom, ..., scorer=...)
    if score >= seuil:
      print(f"{nom:25} -> {correspondance} ({score:.0f})")


def relier(nom: str, seuil: float = 75) -> str | None:
  correspondance, score, _ = process.extractOne(nom, REFERENCE, scorer=fuzz.WRatio)
  regle = ...  # une condition qui écarte le faux lien
  return correspondance if score >= seuil and regle else None


print("--- seuil de 75, avec la règle")
for nom in personnes:
  if relier(nom) is not None:
    print(f"{nom:25} -> {relier(nom)}")
```

### Solution

In [ ]:
from rapidfuzz import process

personnes = sorted(
  {nom for prediction in predictions_gliner for nom in prediction["people"]}
)
for seuil in (85, 75):
  print(f"--- seuil de {seuil}")
  for nom in personnes:
    correspondance, score, _ = process.extractOne(nom, REFERENCE, scorer=fuzz.WRatio)
    if score >= seuil:
      print(f"{nom:25} -> {correspondance} ({score:.0f})")

# À 85, 12 liens, dont un faux : il -> Joseph Caillaux (90), car WRatio donne 90 à un
# fragment (« il » est dans « Caillaux »). À 75, 3 de plus : M. Calmette et M. Cambon,
# justes, et Reymond -> Raymond Poincaré, faux (le docteur Reymond). Baisser le seuil
# rattrape des variantes, et des noms seulement voisins avec. Une règle : le dernier mot
# (le nom de famille) doit ressembler à celui de la référence. Elle écarte les deux faux
# liens (une longueur minimale n'écarterait que « il »), et garde « £1 Asquith », où
# l'OCR a lu « £1 » pour « M. ».


def relier(nom: str, seuil: float = 75) -> str | None:
  correspondance, score, _ = process.extractOne(nom, REFERENCE, scorer=fuzz.WRatio)
  meme_nom = (
    fuzz.ratio(normalise(nom.split()[-1]), normalise(correspondance.split()[-1])) >= 85
  )
  return correspondance if score >= seuil and meme_nom else None


print("--- seuil de 75, avec la règle")
for nom in personnes:
  if relier(nom) is not None:
    print(f"{nom:25} -> {relier(nom)}")

## 5. Comparer, et repérer les inventions

Rassemblez les F1 des trois méthodes dans un tableau. Puis cherchez les **inventions** du LLM : les entités extraites qui n'apparaissent pas dans le texte du passage (mesurées avec `fuzz.partial_ratio`). Y en a-t-il ? Relisez chacune dans son passage. Laquelle des trois méthodes choisiriez-vous pour 10 millions de passages ?

Sans GPU, la cellule suivante remplace les extractions des 5 passages par celles des 40, faites par Qwen3-4B sur GPU avec les consignes de l'exercice 3 (sans la date) : le tableau et la recherche portent alors sur les 40 passages, comme avec un GPU.

**Documentation utile**

- [`pandas.DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [`rapidfuzz.fuzz.partial_ratio`](https://rapidfuzz.github.io/RapidFuzz/Usage/fuzz.html#partial-ratio)

In [ ]:
# Sans GPU, l'exercice 3 n'a extrait que 5 passages : on charge les extractions des
# 40, faites sur GPU par Qwen3-4B avec les mêmes consignes (sans la date).
if not GPU:
  teacher = pd.read_json("teacher-gold.jsonl", lines=True).set_index("id")
  predictions_llm = teacher.loc[gold.id, CHAMPS].to_dict("records")
  resultats["LLM"] = evalue(predictions_llm)
print(len(predictions_llm), "passages extraits par le LLM")

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `pd.DataFrame({nom: table["F1"] for nom, table in resultats.items()})` : une colonne par méthode
- `fuzz.partial_ratio(normalise(entite), normalise(texte))` : la meilleure similarité entre la chaîne courte et un morceau de la longue ; sous 85, l'entité n'est pas dans le texte

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
print(pd.DataFrame({nom: ... for nom, table in resultats.items()}))

for prediction, texte in zip(predictions_llm, gold.text):
  for champ in CHAMPS:
    for entite in prediction[champ]:
      if fuzz.partial_ratio(normalise(...), normalise(...)) < ...:
        print(f"{champ:13} {entite!r} n'est pas dans le texte")
```

### Solution

In [ ]:
print(pd.DataFrame({nom: table["F1"] for nom, table in resultats.items()}))

for prediction, texte in zip(predictions_llm, gold.text):
  for champ in CHAMPS:
    for entite in prediction[champ]:
      if fuzz.partial_ratio(normalise(entite), normalise(texte)) < 85:
        print(f"{champ:13} {entite!r} n'est pas dans le texte")

# Avec les extractions de Qwen3-4B chargées sans GPU, 4 entités sur 257 sont signalées,
# et une seule est vraiment absente du texte (« gouvernement »). « école du Louvre » y
# est, coupée par l'OCR (« Lou- •;r e ») ; « le chancelier » obtient 84,6 (« au
# chancelier ») ; « l'homme » y est (« un homme »), même si ce n'est pas un nom. Sur un
# texte bruité, une mesure par chaînes signale de fausses inventions : chaque
# signalement se relit dans son passage.

# spaCy est de loin le plus rapide, mais ses étiquettes sont fixes et il se trompe sur un
# texte bruité. GLiNER fait ici au moins aussi bien que le LLM sur les personnes et les
# lieux (l'écart reste dans le bruit de 40 passages), pour un coût bien moindre.
# L'atout du LLM est sa souplesse (un schéma, des champs libres, pas
# seulement des entités), pas son exactitude : mesurez sur vos données. Un modèle plus gros
# changerait peut-être ce résultat. À 10 millions de passages, on garderait GLiNER ; pour
# un schéma qu'il ne sait pas remplir, on utiliserait le LLM pour étiqueter des exemples,
# puis on affinerait un petit modèle (TP d'affinage, plus loin).